In [ ]:
# -*- coding: utf-8 -*-
# 注释版说明：本文件与 src/crawler_bilibili.ipynb 同步，用于阅读和答辩讲解。
# 注释不再写“第 N 行”，避免修改代码后行号对不上。

"""
B站弹幕采集脚本：根据 BV 号获取 cid，再优先拉取分段弹幕，旧 XML 弹幕接口作为兜底。
说明：B站页面显示的弹幕总量不等于旧 XML 接口一次可返回的数量；分段接口通常能拿到更多。
输出：data/raw_bilibili_danmu.csv
"""
# 注释：导入这一步需要的模块或类。
import csv
# 注释：导入这一步需要的模块或类。
import math
# 注释：导入这一步需要的模块或类。
import time
# 注释：导入这一步需要的模块或类。
import xml.etree.ElementTree as ET
# 注释：导入这一步需要的模块或类。
from pathlib import Path
# 注释：导入这一步需要的模块或类。
from typing import List, Dict, Optional

# 注释：导入这一步需要的模块或类。
import requests

# 注释：设置项目路径或默认输出文件。
ROOT = Path(__file__).resolve().parents[1]
# 注释：设置项目路径或默认输出文件。
OUT = ROOT / "data" / "raw_bilibili_danmu.csv"

# 注释：设置请求头，用来模拟浏览器访问 B 站接口。
HEADERS = {
    # 注释：执行这行代码对应的核心逻辑。
    "User-Agent": "Mozilla/5.0",
    # 注释：执行这行代码对应的核心逻辑。
    "Referer": "https://www.bilibili.com/",
    # 如遇到403/风控，可将浏览器 Cookie 填在这里：
    # "Cookie": "SESSDATA=...; bili_jct=...;"
# 注释：执行这行代码对应的核心逻辑。
}

# 示例：把每个分区准备若干 BV 号。真实提交时替换为你们选定的视频。
# 注释：提供脚本独立运行时的示例 BV 列表。
VIDEOS = [
    # 注释：执行这行代码对应的核心逻辑。
    {"partition": "学习区", "bvid": "BV1BiDaB1EEN"},
    # 注释：执行这行代码对应的核心逻辑。
    {"partition": "生活区", "bvid": "BV1MDA6zHEFE"},
# 注释：执行这行代码对应的核心逻辑。
]


# 注释：定义函数，将一个子任务封装起来便于复用。
def get_video_info(bvid: str) -> Dict[str, str]:
    # 注释：指定需要请求的 B 站接口地址。
    url = "https://api.bilibili.com/x/web-interface/view"
    # 注释：向 B 站接口发送 GET 请求并等待响应。
    r = requests.get(url, params={"bvid": bvid}, headers=HEADERS, timeout=15)
    # 注释：如果网络请求失败，在这里抛出异常。
    r.raise_for_status()
    # 注释：把接口返回的 JSON 转成 Python 字典。
    data = r.json()
    # 注释：根据条件判断走哪个分支。
    if data.get("code") != 0:
        # 注释：执行这行代码对应的核心逻辑。
        raise RuntimeError(f"获取 cid 失败：{bvid}, 返回={data}")
    # 注释：执行这行代码对应的核心逻辑。
    info = data["data"]
    # 注释：执行这行代码对应的核心逻辑。
    stat = info.get("stat", {})
    # 注释：执行这行代码对应的核心逻辑。
    pages = [
        # 注释：执行这行代码对应的核心逻辑。
        {
            # 注释：执行这行代码对应的核心逻辑。
            "cid": int(page["cid"]),
            # 注释：执行这行代码对应的核心逻辑。
            "part": page.get("part", ""),
            # 注释：执行这行代码对应的核心逻辑。
            "duration": int(page.get("duration", 0) or 0),
        # 注释：执行这行代码对应的核心逻辑。
        }
        # 注释：循环处理列表中的每一项。
        for page in info.get("pages", [])
        # 注释：根据条件判断走哪个分支。
        if page.get("cid")
    # 注释：执行这行代码对应的核心逻辑。
    ]
    # 注释：将本步计算或采集结果返回给调用者。
    return {
        # 注释：执行这行代码对应的核心逻辑。
        "cid": int(info["cid"]),
        # 注释：执行这行代码对应的核心逻辑。
        "title": info.get("title", ""),
        # 注释：执行这行代码对应的核心逻辑。
        "duration": int(info.get("duration", 0) or 0),
        # 注释：执行这行代码对应的核心逻辑。
        "pages": pages,
        # 注释：执行这行代码对应的核心逻辑。
        "view_count": stat.get("view", 0),
        # 注释：执行这行代码对应的核心逻辑。
        "like_count": stat.get("like", 0),
        # 注释：执行这行代码对应的核心逻辑。
        "coin_count": stat.get("coin", 0),
        # 注释：执行这行代码对应的核心逻辑。
        "favorite_count": stat.get("favorite", 0),
        # 注释：执行这行代码对应的核心逻辑。
        "reply_count": stat.get("reply", 0),
        # 注释：执行这行代码对应的核心逻辑。
        "danmaku_count": stat.get("danmaku", 0),
    # 注释：执行这行代码对应的核心逻辑。
    }


# 注释：定义函数，将一个子任务封装起来便于复用。
def get_cid(bvid: str) -> int:
    # 注释：将本步计算或采集结果返回给调用者。
    return int(get_video_info(bvid)["cid"])


# 注释：定义函数，将一个子任务封装起来便于复用。
def _read_varint(data: bytes, pos: int):
    # 注释：执行这行代码对应的核心逻辑。
    result = 0
    # 注释：执行这行代码对应的核心逻辑。
    shift = 0
    # 注释：执行这行代码对应的核心逻辑。
    while pos < len(data):
        # 注释：执行这行代码对应的核心逻辑。
        byte = data[pos]
        # 注释：执行这行代码对应的核心逻辑。
        pos += 1
        # 注释：执行这行代码对应的核心逻辑。
        result |= (byte & 0x7F) << shift
        # 注释：根据条件判断走哪个分支。
        if not byte & 0x80:
            # 注释：将本步计算或采集结果返回给调用者。
            return result, pos
        # 注释：执行这行代码对应的核心逻辑。
        shift += 7
    # 注释：执行这行代码对应的核心逻辑。
    raise ValueError("invalid protobuf varint")


# 注释：定义函数，将一个子任务封装起来便于复用。
def _skip_protobuf_field(data: bytes, pos: int, wire_type: int) -> int:
    # 注释：根据条件判断走哪个分支。
    if wire_type == 0:
        # 注释：执行这行代码对应的核心逻辑。
        _, pos = _read_varint(data, pos)
        # 注释：将本步计算或采集结果返回给调用者。
        return pos
    # 注释：根据条件判断走哪个分支。
    if wire_type == 1:
        # 注释：将本步计算或采集结果返回给调用者。
        return pos + 8
    # 注释：根据条件判断走哪个分支。
    if wire_type == 2:
        # 注释：执行这行代码对应的核心逻辑。
        length, pos = _read_varint(data, pos)
        # 注释：将本步计算或采集结果返回给调用者。
        return pos + length
    # 注释：根据条件判断走哪个分支。
    if wire_type == 5:
        # 注释：将本步计算或采集结果返回给调用者。
        return pos + 4
    # 注释：执行这行代码对应的核心逻辑。
    raise ValueError(f"unsupported protobuf wire type: {wire_type}")


# 注释：定义函数，将一个子任务封装起来便于复用。
def _parse_danmaku_elem_content(elem: bytes) -> str:
    # 注释：执行这行代码对应的核心逻辑。
    pos = 0
    # 注释：执行这行代码对应的核心逻辑。
    while pos < len(elem):
        # 注释：执行这行代码对应的核心逻辑。
        key, pos = _read_varint(elem, pos)
        # 注释：执行这行代码对应的核心逻辑。
        field_no = key >> 3
        # 注释：执行这行代码对应的核心逻辑。
        wire_type = key & 0x07
        # 注释：根据条件判断走哪个分支。
        if field_no == 7 and wire_type == 2:
            # 注释：执行这行代码对应的核心逻辑。
            length, pos = _read_varint(elem, pos)
            # 注释：将本步计算或采集结果返回给调用者。
            return elem[pos:pos + length].decode("utf-8", errors="ignore").strip()
        # 注释：执行这行代码对应的核心逻辑。
        pos = _skip_protobuf_field(elem, pos, wire_type)
    # 注释：将本步计算或采集结果返回给调用者。
    return ""


# 注释：定义函数，将一个子任务封装起来便于复用。
def _parse_segment_reply(data: bytes) -> List[str]:
    """解析 x/v2/dm/web/seg.so 返回的 protobuf，提取 DanmakuElem.content。"""
    # 注释：执行这行代码对应的核心逻辑。
    texts = []
    # 注释：执行这行代码对应的核心逻辑。
    pos = 0
    # 注释：执行这行代码对应的核心逻辑。
    while pos < len(data):
        # 注释：执行这行代码对应的核心逻辑。
        key, pos = _read_varint(data, pos)
        # 注释：执行这行代码对应的核心逻辑。
        field_no = key >> 3
        # 注释：执行这行代码对应的核心逻辑。
        wire_type = key & 0x07
        # 注释：根据条件判断走哪个分支。
        if field_no == 1 and wire_type == 2:
            # 注释：执行这行代码对应的核心逻辑。
            length, pos = _read_varint(data, pos)
            # 注释：执行这行代码对应的核心逻辑。
            text = _parse_danmaku_elem_content(data[pos:pos + length])
            # 注释：根据条件判断走哪个分支。
            if text:
                # 注释：执行这行代码对应的核心逻辑。
                texts.append(text)
            # 注释：执行这行代码对应的核心逻辑。
            pos += length
        # 注释：执行这行代码对应的核心逻辑。
        else:
            # 注释：执行这行代码对应的核心逻辑。
            pos = _skip_protobuf_field(data, pos, wire_type)
    # 注释：将本步计算或采集结果返回给调用者。
    return texts


# 注释：定义函数，将一个子任务封装起来便于复用。
def get_danmu_xml(cid: int) -> List[str]:
    # 注释：指定需要请求的 B 站接口地址。
    url = "https://api.bilibili.com/x/v1/dm/list.so"
    # 注释：向 B 站接口发送 GET 请求并等待响应。
    r = requests.get(url, params={"oid": cid}, headers=HEADERS, timeout=20)
    # 注释：如果网络请求失败，在这里抛出异常。
    r.raise_for_status()
    # 注释：执行这行代码对应的核心逻辑。
    r.encoding = "utf-8"
    # 注释：执行这行代码对应的核心逻辑。
    root = ET.fromstring(r.text)
    # 注释：将本步计算或采集结果返回给调用者。
    return [node.text.strip() for node in root.findall("d") if node.text and node.text.strip()]


# 注释：定义函数，将一个子任务封装起来便于复用。
def get_danmu_segmented(cid: int, duration: int = 0) -> List[str]:
    # 注释：指定需要请求的 B 站接口地址。
    url = "https://api.bilibili.com/x/v2/dm/web/seg.so"
    # 注释：执行这行代码对应的核心逻辑。
    segment_count = max(1, math.ceil(duration / 360)) if duration else 1
    # 注释：执行这行代码对应的核心逻辑。
    texts: List[str] = []
    # 注释：循环处理列表中的每一项。
    for segment_index in range(1, segment_count + 1):
        # 注释：向 B 站接口发送 GET 请求并等待响应。
        r = requests.get(
            # 注释：执行这行代码对应的核心逻辑。
            url,
            # 注释：执行这行代码对应的核心逻辑。
            params={"type": 1, "oid": cid, "segment_index": segment_index},
            # 注释：执行这行代码对应的核心逻辑。
            headers=HEADERS,
            # 注释：执行这行代码对应的核心逻辑。
            timeout=20,
        # 注释：执行这行代码对应的核心逻辑。
        )
        # 注释：如果网络请求失败，在这里抛出异常。
        r.raise_for_status()
        # 注释：执行这行代码对应的核心逻辑。
        texts.extend(_parse_segment_reply(r.content))
        # 注释：执行这行代码对应的核心逻辑。
        time.sleep(0.08)
    # 注释：将本步计算或采集结果返回给调用者。
    return texts


# 注释：定义函数，将一个子任务封装起来便于复用。
def get_danmu(cid: int, duration: int = 0) -> List[str]:
    """优先用分段接口获取更完整弹幕；如果失败或数量更少，则回退到旧 XML 接口。"""
    # 注释：执行这行代码对应的核心逻辑。
    segmented_texts: List[str] = []
    # 注释：执行这行代码对应的核心逻辑。
    xml_texts: List[str] = []
    # 注释：尝试执行可能失败的操作，后面用 except 兜底。
    try:
        # 注释：执行这行代码对应的核心逻辑。
        segmented_texts = get_danmu_segmented(cid, duration)
    # 注释：捕获异常，让程序给出提示后继续运行。
    except Exception as e:
        # 注释：在终端打印进度或结果，方便观察运行情况。
        print(f"分段弹幕接口失败 cid={cid}: {e}")
    # 注释：尝试执行可能失败的操作，后面用 except 兜底。
    try:
        # 注释：执行这行代码对应的核心逻辑。
        xml_texts = get_danmu_xml(cid)
    # 注释：捕获异常，让程序给出提示后继续运行。
    except Exception as e:
        # 注释：在终端打印进度或结果，方便观察运行情况。
        print(f"XML 弹幕接口失败 cid={cid}: {e}")
    # 注释：将本步计算或采集结果返回给调用者。
    return segmented_texts if len(segmented_texts) >= len(xml_texts) else xml_texts


# 注释：定义函数，将一个子任务封装起来便于复用。
def crawl(
    # 注释：执行这行代码对应的核心逻辑。
    videos: Optional[List[Dict[str, str]]] = None,
    # 注释：执行这行代码对应的核心逻辑。
    out: Path = OUT,
    # 注释：执行这行代码对应的核心逻辑。
    max_per_video: Optional[int] = None,
# 注释：执行这行代码对应的核心逻辑。
) -> List[Dict[str, str]]:
    # 注释：执行这行代码对应的核心逻辑。
    videos = videos or VIDEOS
    # 注释：执行这行代码对应的核心逻辑。
    out = Path(out)
    # 注释：执行这行代码对应的核心逻辑。
    out.parent.mkdir(parents=True, exist_ok=True)
    # 注释：执行这行代码对应的核心逻辑。
    rows: List[Dict[str, str]] = []
    # 注释：循环处理列表中的每一项。
    for item in videos:
        # 注释：执行这行代码对应的核心逻辑。
        bvid = item["bvid"]
        # 注释：执行这行代码对应的核心逻辑。
        partition = item["partition"]
        # 注释：尝试执行可能失败的操作，后面用 except 兜底。
        try:
            # 注释：执行这行代码对应的核心逻辑。
            video_info = get_video_info(bvid)
            # 注释：执行这行代码对应的核心逻辑。
            texts: List[str] = []
            # 注释：执行这行代码对应的核心逻辑。
            pages = video_info.get("pages") or [
                # 注释：执行这行代码对应的核心逻辑。
                {"cid": video_info["cid"], "duration": video_info.get("duration", 0), "part": ""}
            # 注释：执行这行代码对应的核心逻辑。
            ]
            # 注释：循环处理列表中的每一项。
            for page in pages:
                # 注释：执行这行代码对应的核心逻辑。
                texts.extend(get_danmu(int(page["cid"]), int(page.get("duration", 0) or 0)))
            # 注释：执行这行代码对应的核心逻辑。
            fetched_count = len(texts)
            # 注释：根据条件判断走哪个分支。
            if max_per_video:
                # 注释：执行这行代码对应的核心逻辑。
                texts = texts[:max_per_video]
            # 注释：循环处理列表中的每一项。
            for text in texts:
                # 注释：执行这行代码对应的核心逻辑。
                rows.append({
                    # 注释：执行这行代码对应的核心逻辑。
                    "partition": partition,
                    # 注释：执行这行代码对应的核心逻辑。
                    "bvid": bvid,
                    # 注释：执行这行代码对应的核心逻辑。
                    "title": video_info["title"],
                    # 注释：执行这行代码对应的核心逻辑。
                    "view_count": video_info["view_count"],
                    # 注释：执行这行代码对应的核心逻辑。
                    "like_count": video_info["like_count"],
                    # 注释：执行这行代码对应的核心逻辑。
                    "coin_count": video_info["coin_count"],
                    # 注释：执行这行代码对应的核心逻辑。
                    "favorite_count": video_info["favorite_count"],
                    # 注释：执行这行代码对应的核心逻辑。
                    "reply_count": video_info["reply_count"],
                    # 注释：执行这行代码对应的核心逻辑。
                    "danmaku_count": video_info["danmaku_count"],
                    # 注释：执行这行代码对应的核心逻辑。
                    "text": text,
                # 注释：执行这行代码对应的核心逻辑。
                })
            # 注释：执行这行代码对应的核心逻辑。
            limit_note = f"，按实验上限保留 {len(texts)} 条" if max_per_video and fetched_count > len(texts) else ""
            # 注释：在终端打印进度或结果，方便观察运行情况。
            print(
                # 注释：执行这行代码对应的核心逻辑。
                f"{partition} {bvid}: 页面弹幕数 {video_info['danmaku_count']}，"
                # 注释：执行这行代码对应的核心逻辑。
                f"实际获取 {fetched_count} 条{limit_note}"
            # 注释：执行这行代码对应的核心逻辑。
            )
            # 注释：根据条件判断走哪个分支。
            if video_info["danmaku_count"] and fetched_count < int(video_info["danmaku_count"]):
                # 注释：在终端打印进度或结果，方便观察运行情况。
                print(
                    # 注释：执行这行代码对应的核心逻辑。
                    "提示：页面弹幕数通常包含累计/历史弹幕，公开接口可能只返回当前可访问弹幕；"
                    # 注释：执行这行代码对应的核心逻辑。
                    "如需更多数据，可换弹幕更多且接口可返回更多的视频，或配置 Cookie 后扩展历史弹幕采集。"
                # 注释：执行这行代码对应的核心逻辑。
                )
            # 注释：执行这行代码对应的核心逻辑。
            time.sleep(1)
        # 注释：捕获异常，让程序给出提示后继续运行。
        except Exception as e:
            # 注释：在终端打印进度或结果，方便观察运行情况。
            print(f"跳过 {bvid}: {e}")

    # 注释：用上下文管理器安全地处理文件读写。
    with out.open("w", newline="", encoding="utf-8-sig") as f:
        # 注释：执行这行代码对应的核心逻辑。
        writer = csv.DictWriter(f, fieldnames=[
            # 注释：执行这行代码对应的核心逻辑。
            "partition", "bvid", "title", "view_count", "like_count",
            # 注释：执行这行代码对应的核心逻辑。
            "coin_count", "favorite_count", "reply_count", "danmaku_count", "text",
        # 注释：执行这行代码对应的核心逻辑。
        ])
        # 注释：执行这行代码对应的核心逻辑。
        writer.writeheader()
        # 注释：执行这行代码对应的核心逻辑。
        writer.writerows(rows)
    # 注释：在终端打印进度或结果，方便观察运行情况。
    print(f"已保存：{out}, 共 {len(rows)} 条")
    # 注释：将本步计算或采集结果返回给调用者。
    return rows


# 注释：根据条件判断走哪个分支。
if __name__ == "__main__":
    # 注释：执行这行代码对应的核心逻辑。
    crawl()
